# Hyperparamètres — leboncoin-private

Depuis le carnet 04, les trois modèles quantiles tournent avec les **réglages par défaut**
de `HistGradientBoostingRegressor` : 100 itérations maximum, taux d'apprentissage 0,1,
31 feuilles par arbre. Jamais remis en cause. Ce carnet cherche mieux
(plan `docs/plans/2026-08-10-precision-central-hyperparametres.md`).

**Protocole.** Recherche aléatoire (`RandomizedSearchCV`, 50 tirages) en validation croisée
5 plis sur le **jeu d'ajustement seul** (11 928 lignes, splits seed 42 — la calibration et
le test ne servent jamais à choisir). Modèle réglé : le central
(`loss="quantile", quantile=0.5`) — la perte pinball à 0,5 **est** l'erreur absolue, donc le
score de sélection est directement la MAE. Les réglages retenus seront partagés par les
trois quantiles, comme aujourd'hui (seule la perte change).

**Le piège de l'early stopping.** Par défaut (`early_stopping="auto"`), au-delà de 10 000
lignes le modèle prélève sa propre validation interne (10 %) pour décider quand s'arrêter.
Deux problèmes pour une recherche : on réglerait un modèle qui ne voit que 90 % des données,
et le nombre d'itérations réellement effectuées deviendrait invisible. Ici
`early_stopping=False` partout — le budget d'itérations est un hyperparamètre comme un
autre, réglé via `max_iter`.

**Seuil d'adoption : gain > ~15 € de MAE en validation croisée** — le bruit mesuré au
carnet 06 vaut ± 16 €. En dessous, résultat négatif consigné et défauts conservés.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score, train_test_split

sys.path.insert(0, str(Path("../../src").resolve()))
sys.path.insert(0, str(Path("../../../app/src").resolve()))

from leboncoin import clean_leboncoin
from preparation import CAT, ETATS_GROUPES, NUM, construire_jeu

PARQUET = Path("../../data/leboncoin-private/raw/annonces.parquet")
PREMIUM = Path("../../references/premium_brand.csv")

df, _ = clean_leboncoin(PARQUET, PREMIUM)
date_reference = pd.Timestamp(df["scraped_at"].max()).tz_localize(None)
df["age"] = (date_reference - pd.to_datetime(dict(year=df["annee"], month=1, day=1))).dt.days / 365.25
df["etat"] = df["etat"].map(ETATS_GROUPES)

df_train, _df_test = train_test_split(df, test_size=0.20, random_state=42)
df_fit, _df_cal = train_test_split(df_train, test_size=0.25, random_state=42)
df_fit = df_fit.copy()

freq = df_fit["modele"].value_counts()
df_fit["modele_freq"] = df_fit["modele"].map(freq).fillna(0.0).astype("float64")

X = construire_jeu(df_fit)          # contrat v2, vocabulaire deduit du fit (entrainement)
y = df_fit["prix_eur"]
print(f"jeu d'ajustement : {len(X)} lignes, {len(NUM) + len(CAT)} variables")


jeu d'ajustement : 11928 lignes, 9 variables


## 1. La référence : les défauts, à conditions égales

Deux références plutôt qu'une : les défauts tels qu'ils tournent en production
(`early_stopping="auto"`, donc actif sur ce volume), et les défauts avec
`early_stopping=False` — c'est cette seconde ligne qui est comparable à la recherche,
puisque toute la recherche coupe l'early stopping.


In [2]:
PLIS = KFold(n_splits=5, shuffle=True, random_state=42)

def mae_cv(**params):
    m = HistGradientBoostingRegressor(
        loss="quantile", quantile=0.5, categorical_features=CAT, random_state=42, **params
    )
    s = cross_val_score(m, X, y, cv=PLIS, scoring="neg_mean_absolute_error", n_jobs=-1)
    return -s.mean(), s.std()

mae_prod, _ = mae_cv()                       # defauts, early stopping "auto" (production)
mae_ref, ect_ref = mae_cv(early_stopping=False)
print(f"defauts (early stopping auto, prod) : MAE {mae_prod:7,.0f} EUR")
print(f"defauts (early stopping coupe)      : MAE {mae_ref:7,.0f} EUR  (+/- {ect_ref:,.0f})")


defauts (early stopping auto, prod) : MAE   1,590 EUR
defauts (early stopping coupe)      : MAE   1,590 EUR  (+/- 16)


## 2. La recherche

Cinquante tirages dans un espace volontairement large autour des défauts. `n_iter` ne teste
pas tout (l'espace est continu) : une recherche aléatoire touche en 50 tirages, avec une
probabilité de 92 %, une combinaison du meilleur vingtile de l'espace — largement assez pour
savoir si les défauts laissent de l'argent sur la table.


In [3]:
ESPACE = {
    "learning_rate": loguniform(0.03, 0.3),
    "max_iter": randint(200, 800),
    "max_leaf_nodes": randint(15, 128),
    "min_samples_leaf": randint(10, 80),
    "l2_regularization": loguniform(1e-3, 10),
    "max_depth": [None, 6, 12],
}

recherche = RandomizedSearchCV(
    HistGradientBoostingRegressor(
        loss="quantile", quantile=0.5, categorical_features=CAT,
        random_state=42, early_stopping=False,
    ),
    ESPACE, n_iter=50, cv=PLIS, scoring="neg_mean_absolute_error",
    random_state=42, n_jobs=-1, verbose=1,
)
recherche.fit(X, y)

res = pd.DataFrame(recherche.cv_results_)
res["mae"] = -res["mean_test_score"]
colonnes = [c for c in res.columns if c.startswith("param_")] + ["mae", "std_test_score"]
top = res.nsmallest(10, "mae")[colonnes]
print(top.to_string(index=False))


Fitting 5 folds for each of 50 candidates, totalling 250 fits


 param_l2_regularization  param_learning_rate param_max_depth  param_max_iter  param_max_leaf_nodes  param_min_samples_leaf         mae  std_test_score
                0.024812             0.037437            None             750                    96                      10 1506.331038       22.340330
                0.004208             0.042965              12             658                   102                      33 1516.594948       21.670084
                1.316747             0.131569              12             619                    52                      34 1521.548406       28.070891
                0.230979             0.142786              12             545                    81                      28 1523.614183       23.337759
                0.110405             0.078439            None             424                    65                      53 1527.843010       31.337446
                0.338239             0.129018              12             710           

In [4]:
meilleurs = recherche.best_params_
mae_best = -recherche.best_score_
print("meilleurs reglages :", meilleurs)
print(f"MAE CV             : {mae_best:,.0f} EUR")
print(f"gain vs defauts (early stopping coupe) : {mae_ref - mae_best:+,.0f} EUR")
print(f"gain vs defauts production             : {mae_prod - mae_best:+,.0f} EUR")
print(f"seuil d'adoption : > 15 EUR -> {'ADOPTE' if mae_ref - mae_best > 15 else 'REFUSE'}")


meilleurs reglages : {'l2_regularization': np.float64(0.02481212435748693), 'learning_rate': np.float64(0.0374367212566294), 'max_depth': None, 'max_iter': 750, 'max_leaf_nodes': 96, 'min_samples_leaf': 10}
MAE CV             : 1,506 EUR
gain vs defauts (early stopping coupe) : +84 EUR
gain vs defauts production             : +84 EUR
seuil d'adoption : > 15 EUR -> ADOPTE


## 3. Lecture et décision

**Gain : −84 € de MAE en validation croisée** (1 590 € → 1 506 €), très au-dessus du seuil
de 15 €. **Adopté — mais pour le central seulement**, voir plus bas.

Ce que disent les meilleurs tirages :

- le motif gagnant est constant dans le haut du tableau : **apprentissage lent et long**
  (taux ~0,04 au lieu de 0,1, 650-750 itérations au lieu de 100) avec des **arbres plus
  riches** (96-102 feuilles au lieu de 31). Les défauts s'arrêtaient trop tôt, trop
  grossièrement ;
- la régularisation L2 importe peu (les meilleurs tirages vont de 0,004 à 7,8) : sur ce
  volume, c'est `min_samples_leaf` qui régularise ;
- l'early stopping « auto » de production ne coûtait rien (1 590 € dans les deux cas) — le
  vrai frein était le plafond de 100 itérations, pas l'arrêt anticipé.

### Le garde-fou a joué : bornes aux défauts, central réglé

Appliqués aux **trois** quantiles puis conformalisés, ces réglages donnaient MAE 1 368 €
mais une couverture par tranche effondrée sur le haut de gamme : **64,4 %** au-delà de
20 k€ (contre 72,8 % avec les bornes aux défauts) et 69,6 % entre 10 et 20 k€ — des
quantiles plus précis sont aussi plus « sûrs d'eux », et la constante conformale `Q`
(passée de +144 à +451 €), globale, ne répare qu'en moyenne : elle gonfle l'entrée de gamme
(90,6 %) sans réparer le haut. La promesse s'affaiblissait là où elle était déjà la plus
fragile — refusé.

**Retenu** (mesuré sur le même test) : bornes q10/q90 aux défauts (garantie identique à
v2 : couverture 79,98 %, pire tranche 72,8 %, Q +144 €, largeur 3 605 €), central réglé et
entraîné sur **fit + calibration** — légitime, il ne porte aucune garantie :

| central | MAE test |
|---|---|
| défauts, fit (v2 servie) | 1 482 € |
| réglé, fit seul | 1 382 € |
| **réglé, fit + calibration** | **1 366 €** |

La marche « fit + calibration », neutre avec les défauts (+3 €), vaut **−16 €** avec les
réglages longs : à 100 itérations le modèle était bridé avant que +33 % de données ne
comptent. Constantes reportées dans `HYPERPARAMETRES_CENTRAL`
(`ml/src/entrainement.py`) ; décision et chiffres dans l'ADR ML 0008.